# Adaptive (probe-and-commit) Stage-3 analysis

Compares the adaptive `--target-k` run against the classic 7-MEL baseline.

- **Classic baseline**: `results/batch_manifest.tsv` + per-MEL `enumerated.sdf` (full library docked for every MEL, ~54 min aggregate).
- **Adaptive run**: `results_local_macos/adaptive/batch_manifest.tsv` + per-MEL `enumerated.sdf` (`--hit-threshold 10 --min-expected-hits 20 --target-k 50`, ~11 min aggregate).

Reuses the helper functions in [analyze_adaptive.py](analyze_adaptive.py) so metrics match the TSV/PNG artifacts written by that script.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import analyze_adaptive as aa

CLASSIC = Path('results/batch_manifest.tsv')
ADAPT   = Path('results_local_macos/adaptive/batch_manifest.tsv')
HIT_THR = 10.0

## Manifest join

In [ ]:
classic = aa._load_manifest(CLASSIC)
adapt   = aa._load_manifest(ADAPT)
joined  = aa._join_manifests(classic, adapt)

pd.DataFrame([{
    'row': m.row, 'rank': m.rank,
    'classic_status': m.classic_status,
    'adapt_status':   m.adapt_status,
    'adapt_decision': m.adapt_decision,
    'n_total':        m.n_total,
    'n_probe':        m.n_probe,
    'elapsed_s':      m.adapt_elapsed_s,
} for m in joined])

## Per-MEL comparison

For each MEL with both classic and adaptive outputs, compute overlap, top-N recovery, score parity, and missed-hits-at-threshold.

In [ ]:
comparisons = [c for c in (aa.compare_one(m, HIT_THR) for m in joined) if c is not None]

df = pd.DataFrame([{
    'row': c.row, 'rank': c.rank, 'decision': c.adapt_decision,
    'n_classic': c.n_classic, 'n_adapt': c.n_adaptive,
    'overlap_pct': c.overlap_pct,
    'missing': c.missing_in_adapt, 'extra': c.extra_in_adapt,
    'classic_top1': c.classic_top1, 'adapt_top1': c.adaptive_top1,
    'top10':  c.top10_recovery,
    'top100': c.top100_recovery,
    'top1k':  c.top1000_recovery,
    'diff_p95': c.score_diff_p95,
    f'missed>={HIT_THR:g}': c.missed_hits_at_thr,
    'elapsed_s': c.adapt_elapsed_s,
} for c in comparisons])
df.style.format({
    'overlap_pct': '{:.1f}', 'classic_top1': '{:.2f}', 'adapt_top1': '{:.2f}',
    'top10': '{:.2f}', 'top100': '{:.2f}', 'top1k': '{:.2f}',
    'diff_p95': '{:.2f}', 'elapsed_s': '{:.0f}',
})

## Figure: recovery, timings, top1, score diffs

Regenerates `adaptive_vs_classic.png` via `aa.write_png` and displays inline.

In [ ]:
from IPython.display import Image

png_path = Path('results_local_macos/adaptive/adaptive_vs_classic.png')
aa.write_png(comparisons, png_path, HIT_THR)
Image(filename=str(png_path))

## Interpretation

**Time saving is real.** Probe-only MELs (5/6/9) dock 5% of the library; aborts (3/10) stop after the probe. Commit phase only fires for low-density MELs (2/8), and even there the commit subsample is capped to reach `target_k=50` hits in expectation.

**Top-K recovery is lossy.** The target-K design assumes hit density is uniform across the library — so a random commit subsample finds hits in proportion to probe density. But classic's top-10 RTCNN candidates are tail outliers, not density-representative:

- **Probe-only MELs**: top-10 recovery 0.10–0.20. The 95% we discarded contained thousands of hits above `RTCNN ≥ 10`, including classic's actual top ranks. `adaptive_top1` for these MELs is the best score in the 5% probe, not close to classic top1.
- **Committed MELs**: even with large commit budgets (MEL_2 ≈ 53% of library, MEL_8 ≈ 83%), top-10 recovery is 0.50 / 0.70 — random sampling can't guarantee outlier preservation.
- **Aborts**: correct. MEL_3 has 0 classic hits ≥10; MEL_10 has 1 that the probe missed (acceptable).

### Mismatch with Stage-4 semantics

Stage-4 selects O(10–100) molecules by top-K RTCNN rank, further filtered by full-ligand docking scores < −25 or < −30. That workflow needs actual top RTCNN ranks preserved — not a density-matched sample. Target-K optimizes for the wrong quantity.

### Options going forward

1. **Revert to probe-gate only.** Keep abort logic; when probe passes, dock the full remainder. Saves time on aborted MELs only (2/7 → ~10 min saving). Recovers all classic top hits on passing MELs by construction.
2. **Raise `target_k` substantially** (e.g. 500 or 1000). Trades saving for recovery; probe-only MELs would still miss outliers in the unsampled portion.
3. **Replace random commit with RTCNN-guided selection.** Requires a proxy model — out of scope for this pilot.

Recommendation: option 1 for near-term Stage-4 handoff; revisit target-K only if Stage-4 tolerates the missed-hit rates above.